# 01 — Data Inspection

Purpose: verify the raw transfermarkt-datasets tables before doing any
cleaning or analysis. No aggregation, no visualization of results here —
just confirming row counts, columns, date coverage, missingness, and join
keys match what's documented in `DATA_PLAN.md`.

Assumes the raw CSVs have been downloaded per `data/raw/README.md`.

## Imports

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
pd.set_option("display.max_columns", 50)

## Load raw tables

In [2]:
players = pd.read_csv(RAW / "players.csv.gz", low_memory=False)
appearances = pd.read_csv(RAW / "appearances.csv.gz", low_memory=False)
player_valuations = pd.read_csv(RAW / "player_valuations.csv.gz", low_memory=False)
games = pd.read_csv(RAW / "games.csv.gz", low_memory=False)
clubs = pd.read_csv(RAW / "clubs.csv.gz", low_memory=False)
competitions = pd.read_csv(RAW / "competitions.csv.gz", low_memory=False)

tables = {
    "players": players,
    "appearances": appearances,
    "player_valuations": player_valuations,
    "games": games,
    "clubs": clubs,
    "competitions": competitions,
}

## Row counts and columns

In [3]:
for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(list(df.columns))
    print()

players: 50,149 rows x 26 cols
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

appearances: 1,894,350 rows x 13 cols
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']

player_valuations: 656,301 rows x 6 cols
['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']

games: 88,958 rows x 23 cols
['game_id', 'competition_id', 'season', 'round', 'date', 'ho

## Date ranges

In [4]:
print("games.season:", games["season"].min(), "-", games["season"].max())
print("games.date:", games["date"].min(), "-", games["date"].max())
print("appearances.date:", appearances["date"].min(), "-", appearances["date"].max())
print("player_valuations.date:", player_valuations["date"].min(), "-", player_valuations["date"].max())

games.season: 2005 - 2025
games.date: 2006-06-09 - 2026-07-06


appearances.date: 2012-07-03 - 2026-06-28
player_valuations.date: 2000-01-20 - 2026-06-12


In [5]:
games["season"].value_counts().sort_index()

season
2005       64
2007       31
2009       64
2011       31
2012     5700
2013     5826
2014     5836
2015     5759
2016     5699
2017     5660
2018     5723
2019     5469
2020     5564
2021     5939
2022     6008
2023     5951
2024    10036
2025     9598
Name: count, dtype: int64

## Missing values

In [6]:
for name, df in tables.items():
    missing = df.isna().mean().sort_values(ascending=False)
    print(f"--- {name} (top missing columns) ---")
    print(missing[missing > 0].head(10))
    print()

--- players (top missing columns) ---
current_national_team_id       0.934994
international_goals            0.614030
international_caps             0.614030
agent_name                     0.464536
contract_expiration_date       0.370277
highest_market_value_in_eur    0.171908
market_value_in_eur            0.171908
foot                           0.117510
country_of_birth               0.117350
city_of_birth                  0.111827
dtype: float64



--- appearances (top missing columns) ---
player_name    0.000001
dtype: float64

--- player_valuations (top missing columns) ---
player_club_domestic_competition_id    0.141662
dtype: float64

--- games (top missing columns) ---
home_club_position        0.285269
away_club_position        0.285269
attendance                0.121439
home_club_formation       0.092178
away_club_formation       0.090751
competition_type          0.013647
away_club_manager_name    0.009488
home_club_manager_name    0.009488
referee                   0.007622
stadium                   0.002720
dtype: float64

--- clubs (top missing columns) ---
total_market_value       1.000000
coach_name               0.493719
foreigners_percentage    0.075377
average_age              0.047739
dtype: float64

--- competitions (top missing columns) ---
total_clubs             0.230769
country_name            0.184615
domestic_league_code    0.184615
dtype: float64



## Duplicate checks on expected keys

In [7]:
print("players.player_id unique:", players["player_id"].is_unique)
print("games.game_id unique:", games["game_id"].is_unique)
print("appearances.appearance_id unique:", appearances["appearance_id"].is_unique)
print(
    "player_valuations (player_id, date) unique:",
    not player_valuations.duplicated(subset=["player_id", "date"]).any(),
)

players.player_id unique: True
games.game_id unique: True


appearances.appearance_id unique: True
player_valuations (player_id, date) unique: True


## Sample records

In [8]:
players.sample(5, random_state=0)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,date_of_birth,sub_position,position,foot,height_in_cm,contract_expiration_date,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
23762,396823,Viktor,Boone,Viktor Boone,2023,3948,viktor-boone,NaN,NaN,Belgium,1998-01-25 00:00:00,Centre-Back,Defender,right,191.0,2026-06-30 00:00:00,Stirr Associates,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/viktor-boone/p...,BE1,Union Saint-Gilloise,350000.0,400000.0
1948,22969,NaN,Gabi,Gabi,2017,13,gabi,Spain,Madrid,Spain,1983-07-10 00:00:00,Central Midfield,Midfield,right,181.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/gabi/profil/sp...,ES1,Atlético de Madrid,1100000.0,12000000.0
44225,1061690,Freddie,Owens,Freddie Owens,2025,903,freddie-owens,NaN,NaN,Scotland,2005-09-28 00:00:00,Goalkeeper,Goalkeeper,NaN,NaN,2026-05-31 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/freddie-owens/...,SC1,Hibernian FC,NaN,NaN
6655,76701,Diego,Ifrán,Diego Ifrán,2013,681,diego-ifran,Uruguay,Montevideo,Uruguay,1987-06-08 00:00:00,Centre-Forward,Attack,right,174.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/diego-ifran/pr...,ES1,Real Sociedad,675000.0,2000000.0
18636,294322,Pedro,Empis,Pedro Empis,2025,33328,pedro-empis,Portugal,Estoril,Portugal,1997-02-01 00:00:00,Left-Back,Defender,left,177.0,NaN,Prodigy Players,https://img.a.transfermarkt.technology/portrai...,1.0,0.0,NaN,https://www.transfermarkt.co.uk/pedro-empis/pr...,SER1,FK Radnik Surdulica,NaN,NaN


In [9]:
appearances.sample(5, random_state=0)

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
832215,3082146_213865,3082146,213865,338,48332,2018-11-11,Mykyta Burda,UKR1,0,0,0,0,90
747536,2871761_32711,2871761,32711,18,18,2018-02-24,Patrick Herrmann,L1,0,0,0,0,6
692030,2897241_195810,2897241,195810,398,537,2017-10-22,Bastos,IT1,0,0,1,0,90
126390,2244043_170644,2244043,170644,2381,2381,2013-05-13,Ziya Erdal,TR1,0,0,0,0,90
1327650,3851265_337670,3851265,337670,354,3899,2022-09-10,Alessio Da Cruz,BE1,1,0,0,0,75


In [10]:
player_valuations.sample(5, random_state=0)

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
21990,40059,2008-12-23,3500000,Rosenborg BK,524,TR1
571792,336074,2024-10-04,900000,Samsunspor,152,TR1
81,1606,2004-10-04,150000,Bröndby IF U19,2414,DK1
84825,202959,2012-12-06,50000,Figueirense Futebol Clube,4064,NaN
181975,113045,2016-07-15,2500000,Granada CF,16795,ES1


## Join checks: appearances -> games -> season

In [11]:
appearances_with_season = appearances.merge(
    games[["game_id", "season", "competition_type"]], on="game_id", how="left"
)
print("unmatched game_id rate:", appearances_with_season["season"].isna().mean())
appearances_with_season["season"].value_counts().sort_index()

unmatched game_id rate: 0.0


season
2012    130736
2013    131641
2014    127546
2015    131436
2016    131655
2017    128457
2018    129455
2019    122825
2020    137720
2021    131632
2022    143562
2023    142869
2024    146972
2025    157844
Name: count, dtype: int64

## Competition / league coverage

In [12]:
competitions["type"].value_counts()

type
domestic_league              31
other                        16
domestic_cup                 10
national_team_competition     5
international_cup             3
Name: count, dtype: int64

In [13]:
appearances_with_season[
    appearances_with_season["competition_type"] == "domestic_league"
]["season"].value_counts().sort_index()

season
2012    111786
2013    113376
2014    109008
2015    112133
2016    112617
2017    110097
2018    109867
2019    104425
2020    120679
2021    114328
2022    123335
2023    122153
2024    121733
2025    120565
Name: count, dtype: int64

## Player position counts

In [14]:
players["position"].value_counts(dropna=False)

position
Defender      15961
Midfield      14272
Attack        13502
Goalkeeper     5828
Missing         586
Name: count, dtype: int64

In [15]:
players["sub_position"].value_counts(dropna=False)

sub_position
Centre-Back           8766
Centre-Forward        6749
Goalkeeper            5828
Central Midfield      5498
Defensive Midfield    4123
Right-Back            3700
Attacking Midfield    3520
Left-Back             3495
Left Winger           3297
Right Winger          3170
NaN                    586
Left Midfield          566
Right Midfield         565
Second Striker         286
Name: count, dtype: int64

## Season and competition-type coverage

Before deciding the analysis scope: how does `games.season` behave, which
competitions are domestic leagues vs. cups/international, and how much data
is there per season/competition type.

In [16]:
app_full = appearances.merge(
    games[["game_id", "season", "competition_id", "date"]],
    on="game_id", how="left", suffixes=("_appearance", "")
).merge(games[["game_id", "competition_type"]], on="game_id", how="left")

season_x_type = pd.crosstab(app_full["season"], app_full["competition_type"])
season_x_type.tail(12)

competition_type,domestic_cup,domestic_league,international_cup,national_team_competition,other
season,,,,,
2014,8007,109008,3297,0,6154
2015,8439,112133,3763,0,6338
2016,8050,112617,3930,0,6176
2017,8115,110097,3886,0,5497
2018,8571,109867,3887,0,6235
2019,7597,104425,3643,0,6312
2020,7789,120679,3909,0,5098
2021,7238,114328,4082,0,5334
2022,8791,123335,4048,0,5287


`games.season` is a direct integer column — no need to derive it from raw
dates. Confirmed values run 2005-2025. `season=2023` denotes the 2023/24
season for leagues that run Aug-May; see the calendar-convention check below
for leagues where that isn't true.

### Domestic leagues vs. other competition types

In [17]:
competitions["type"].value_counts()

type
domestic_league              31
other                        16
domestic_cup                 10
national_team_competition     5
international_cup             3
Name: count, dtype: int64

### Season calendar convention per domestic league

Most European leagues run August (season *N*) through May (season *N*+1).
A handful of leagues (Brazil, MLS, J1 League, Nordic leagues, etc.) instead
run within a single calendar year. Mixing both conventions under one
season-end cutoff rule would be inconsistent, so classify each league by
the typical start month of its season.

In [18]:
domestic_leagues = competitions[competitions["type"] == "domestic_league"]
league_games = games.merge(domestic_leagues[["competition_id", "name", "country_name"]], on="competition_id")
league_games["date"] = pd.to_datetime(league_games["date"])

season_start_month = (
    league_games.groupby(["competition_id", "season"])["date"].min().dt.month
    .groupby("competition_id").median()
)
euro_calendar_ids = season_start_month[season_start_month >= 6].index
calendar_year_ids = season_start_month[season_start_month < 6].index

print("Aug-May (European) calendar leagues:", len(euro_calendar_ids))
print("Calendar-year leagues:", len(calendar_year_ids))
domestic_leagues[domestic_leagues["competition_id"].isin(calendar_year_ids)][["competition_id", "name", "country_name"]]

Aug-May (European) calendar leagues: 23
Calendar-year leagues: 8


,competition_id,name,country_name
3,ARG1,torneo-apertura,Argentina
8,BRA1,campeonato-brasileiro-serie-a,Brazil
34,JAP1,j1-league,Japan
37,MEX1,liga-mx-clausura,Mexico
38,MLS1,major-league-soccer,United States
42,NO1,eliteserien,Norway
48,RSK1,k-league-1,"Korea, South"
55,SE1,allsvenskan,Sweden


The calendar-year leagues (Argentina, Brazil, Japan, Mexico, MLS, Norway,
South Korea, Sweden) are excluded from the primary analysis scope — see
`DATA_PLAN.md` for the reasoning. Everything below evaluates coverage for
the remaining Aug-May domestic leagues only.

### Per-season coverage for candidate recent seasons

In [19]:
euro_app = appearances.drop(columns=["competition_id"]).merge(
    games[["game_id", "season", "competition_id"]], on="game_id", how="left"
)
euro_app = euro_app[euro_app["competition_id"].isin(euro_calendar_ids)]

rows = []
for s in range(2017, 2026):
    sub = euro_app[euro_app["season"] == s]
    uniq_players = sub["player_id"].unique()
    pinfo = players[players["player_id"].isin(uniq_players)]
    cutoff = pd.Timestamp(year=s + 1, month=7, day=31)
    valued = player_valuations[
        (player_valuations["player_id"].isin(uniq_players))
        & (pd.to_datetime(player_valuations["date"]) <= cutoff)
    ]["player_id"].nunique()
    rows.append({
        "season": s,
        "unique_players": len(uniq_players),
        "appearance_rows": len(sub),
        "position_valid": int((pinfo["position"] != "Missing").sum()),
        "dob_valid": int(pinfo["date_of_birth"].notna().sum()),
        "players_with_valuation": valued,
        "n_leagues": sub["competition_id"].nunique(),
    })

coverage = pd.DataFrame(rows)
coverage["valuation_coverage_pct"] = (coverage["players_with_valuation"] / coverage["unique_players"]).round(3)
coverage

,season,unique_players,appearance_rows,position_valid,dob_valid,players_with_valuation,n_leagues,valuation_coverage_pct
0,2017,6312,110097,6303,6309,6242,14,0.989
1,2018,6255,109867,6248,6252,6210,14,0.993
2,2019,6381,104425,6377,6380,6287,14,0.985
3,2020,6765,120679,6760,6764,6693,14,0.989
4,2021,5862,114328,5862,5862,5860,14,1.000
5,2022,6803,123335,6802,6802,6711,14,0.986
6,2023,6772,122153,6771,6770,6687,14,0.987
7,2024,6764,121733,6762,6763,6522,14,0.964
8,2025,6738,120565,6735,6738,6430,14,0.954


Coverage is stable across 2017-2025: 5,800-6,800 players/season, 14
leagues every season, position/DOB essentially complete, and 96%+ of
players have a valuation on or before the season-end cutoff. Season
quality is not the limiting factor for any of these years — the choice of
range is about keeping the project scope manageable, not data availability.
See `DATA_PLAN.md` for the final season-range decision.

### Position and league breakdown, most recent complete season (2024/25)

In [20]:
recent_players = euro_app[euro_app["season"] == 2024]["player_id"].unique()
players[players["player_id"].isin(recent_players)]["position"].value_counts()

position
Defender      2285
Midfield      1994
Attack        1970
Goalkeeper     513
Missing          2
Name: count, dtype: int64

In [21]:
recent_by_league = (
    euro_app[euro_app["season"] == 2024]
    .merge(domestic_leagues[["competition_id", "name", "country_name"]], on="competition_id")
    .groupby(["country_name", "name"])["player_id"].nunique()
    .sort_values(ascending=False)
)
recent_by_league

country_name  name                
Italy         serie-a                 599
Spain         laliga                  589
Türkiye       super-lig               582
Portugal      liga-portugal           568
England       premier-league          562
France        ligue-1                 542
Netherlands   eredivisie              519
Ukraine       premier-liga            490
Germany       bundesliga              481
Belgium       jupiler-pro-league      475
Russia        premier-liga            466
Greece        super-league-1          411
Scotland      scottish-premiership    363
Denmark       superliga               336
Name: player_id, dtype: int64

## Next steps

Findings from this notebook feed back into `DATA_PLAN.md`
(competition scope, season range, valuation-to-season mapping). Player-season
construction happens in a later notebook, not here.